In [1]:
import duckdb
con = duckdb.connect("olist.duckdb")
print("connected")

connected


In [2]:
con.execute("""
SELECT
  COUNT(*) AS items,
  SUM(CASE WHEN category IS NULL AND product_category_name IS NULL THEN 1 ELSE 0 END) AS no_category,
  SUM(CASE WHEN category IS NULL AND product_category_name IS NOT NULL THEN 1 ELSE 0 END) AS no_translation
FROM item_level
WHERE order_status = 'delivered'
""").df()

,items,no_category,no_translation
0,110197,1537.0,22.0


In [3]:
con.execute("""
CREATE OR REPLACE TABLE category_economics AS
WITH base AS (
  SELECT COALESCE(category, product_category_name, 'unknown') AS category,
         order_id, price, freight_value
  FROM item_level
  WHERE order_status = 'delivered'
),
agg AS (
  SELECT category,
         COUNT(DISTINCT order_id) AS orders,
         COUNT(*) AS items,
         ROUND(SUM(price), 2) AS revenue,
         ROUND(SUM(freight_value), 2) AS freight,
         ROUND(100.0 * SUM(freight_value) / SUM(price), 2) AS freight_pct_of_price,
         ROUND(AVG(price), 2) AS avg_price,
         ROUND(AVG(freight_value), 2) AS avg_freight
  FROM base
  GROUP BY 1
  HAVING COUNT(DISTINCT order_id) >= 100
)
SELECT *,
       ROUND(100.0 * revenue / (SELECT SUM(price) FROM base), 2) AS revenue_share_pct
FROM agg
ORDER BY revenue DESC
""")

con.execute("SELECT * FROM category_economics").df()

,category,orders,items,revenue,freight,freight_pct_of_price,avg_price,avg_freight,revenue_share_pct
0,health_beauty,8647,9465,1233131.72,178957.81,14.51,130.28,18.91,9.33
1,watches_gifts,5495,5859,1166176.98,98156.14,8.42,199.04,16.75,8.82
2,bed_bath_table,9272,10953,1023434.76,201774.50,19.72,93.44,18.42,7.74
3,sports_leisure,7530,8431,954852.55,163404.36,17.11,113.25,19.38,7.22
4,computers_accessories,6530,7644,888724.61,143999.16,16.20,116.26,18.84,6.72
5,furniture_decor,6307,8160,711927.69,168402.23,23.65,87.25,20.64,5.38
6,housewares,5743,6795,615628.69,142763.56,23.19,90.60,21.01,4.66
7,cool_stuff,3559,3718,610204.10,81476.79,13.35,164.12,21.91,4.62
8,auto,3810,4140,578966.65,90488.10,15.63,139.85,21.86,4.38
9,toys,3804,4030,471286.48,75774.58,16.08,116.94,18.80,3.56


In [4]:
con.execute("""
CREATE OR REPLACE TABLE overall_freight AS
SELECT ROUND(SUM(price), 2) AS revenue,
       ROUND(SUM(freight_value), 2) AS freight,
       ROUND(100.0 * SUM(freight_value) / SUM(price), 2) AS freight_pct_of_price
FROM item_level
WHERE order_status = 'delivered'
""")

con.execute("SELECT * FROM overall_freight").df()

,revenue,freight,freight_pct_of_price
0,13221498.11,2198275.64,16.63


In [5]:
con.execute("""
CREATE OR REPLACE TABLE category_watchlist AS
SELECT *
FROM category_economics
WHERE freight_pct_of_price > (SELECT freight_pct_of_price FROM overall_freight)
  AND revenue >= (SELECT MEDIAN(revenue) FROM category_economics)
ORDER BY freight DESC
""")

con.execute("SELECT * FROM category_watchlist").df()

,category,orders,items,revenue,freight,freight_pct_of_price,avg_price,avg_freight,revenue_share_pct
0,bed_bath_table,9272,10953,1023434.76,201774.50,19.72,93.44,18.42,7.74
1,furniture_decor,6307,8160,711927.69,168402.23,23.65,87.25,20.64,5.38
2,sports_leisure,7530,8431,954852.55,163404.36,17.11,113.25,19.38,7.22
3,housewares,5743,6795,615628.69,142763.56,23.19,90.60,21.01,4.66
4,garden_tools,3448,4268,470495.28,96650.40,20.54,110.24,22.65,3.56
5,telephony,4093,4430,309860.23,69342.39,22.38,69.95,15.65,2.34
6,office_furniture,1254,1668,268154.31,67057.05,25.01,160.76,40.20,2.03
7,stationery,2264,2466,223788.69,45786.36,20.46,90.75,18.57,1.69
8,electronics,2517,2729,155043.93,45679.16,29.46,56.81,16.74,1.17
9,pet_shop,1688,1924,211695.64,38918.56,18.38,110.03,20.23,1.60


In [6]:
con.execute("""
SELECT
  (SELECT ROUND(SUM(price), 2) FROM item_level WHERE order_status = 'delivered') AS item_level_delivered_revenue,
  (SELECT ROUND(SUM(item_revenue), 2) FROM order_level WHERE order_status = 'delivered') AS order_level_delivered_revenue,
  (SELECT ROUND(SUM(revenue), 2) FROM category_economics) AS revenue_in_category_table
""").df()

,item_level_delivered_revenue,order_level_delivered_revenue,revenue_in_category_table
0,13221498.11,13221498.11,13083222.17


In [7]:
print("Top 10 categories by revenue")
display(con.execute("""
SELECT category, orders, revenue, revenue_share_pct, freight_pct_of_price, avg_price
FROM category_economics ORDER BY revenue DESC LIMIT 10
""").df())

print("Top 10 categories by freight share of price")
display(con.execute("""
SELECT category, orders, revenue, freight_pct_of_price, avg_price, avg_freight
FROM category_economics ORDER BY freight_pct_of_price DESC LIMIT 10
""").df())

Top 10 categories by revenue


,category,orders,revenue,revenue_share_pct,freight_pct_of_price,avg_price
0,health_beauty,8647,1233131.72,9.33,14.51,130.28
1,watches_gifts,5495,1166176.98,8.82,8.42,199.04
2,bed_bath_table,9272,1023434.76,7.74,19.72,93.44
3,sports_leisure,7530,954852.55,7.22,17.11,113.25
4,computers_accessories,6530,888724.61,6.72,16.20,116.26
5,furniture_decor,6307,711927.69,5.38,23.65,87.25
6,housewares,5743,615628.69,4.66,23.19,90.60
7,cool_stuff,3559,610204.10,4.62,13.35,164.12
8,auto,3810,578966.65,4.38,15.63,139.85
9,toys,3804,471286.48,3.56,16.08,116.94


Top 10 categories by freight share of price


,category,orders,revenue,freight_pct_of_price,avg_price,avg_freight
0,christmas_supplies,125,8737.84,36.52,58.25,21.27
1,signaling_and_security,138,21315.05,30.39,108.20,32.88
2,electronics,2517,155043.93,29.46,56.81,16.74
3,food_drink,221,14942.88,29.41,55.55,16.34
4,furniture_living_room,414,67350.70,26.27,136.06,35.75
5,kitchen_dining_laundry_garden_furniture,241,45531.98,25.35,166.18,42.12
6,drinks,287,21529.84,25.27,59.64,15.07
7,office_furniture,1254,268154.31,25.01,160.76,40.20
8,food,441,28731.15,24.58,57.58,14.16
9,furniture_decor,6307,711927.69,23.65,87.25,20.64


In [8]:
con.execute("COPY category_economics TO 'dashboard/category_economics.csv' (HEADER, DELIMITER ',')")
con.execute("COPY category_watchlist TO 'dashboard/category_watchlist.csv' (HEADER, DELIMITER ',')")
con.execute("COPY overall_freight TO 'dashboard/overall_freight.csv' (HEADER, DELIMITER ',')")
print("saved")

saved


In [9]:
e = con.execute("SELECT * FROM category_economics").df()
o = con.execute("SELECT * FROM overall_freight").df().iloc[0]
w = con.execute("SELECT * FROM category_watchlist").df()

In [10]:
top_rev = e.nlargest(3, "revenue")
top_fr = e.nlargest(3, "freight_pct_of_price")

rev_txt = ", ".join(f"{r.category} ({r.revenue_share_pct}% of revenue, freight {r.freight_pct_of_price}% of price)" for r in top_rev.itertuples())
fr_txt = ", ".join(f"{r.category} ({r.freight_pct_of_price}% of price, average item price {r.avg_price})" for r in top_fr.itertuples())

if len(w) > 0:
    w3 = w.nlargest(3, "freight")
    w_txt = ", ".join(f"{r.category} (revenue {r.revenue:,.0f}, freight {r.freight_pct_of_price}% of price)" for r in w3.itertuples())
    w_sentence = f"{len(w)} categories are both above-median in revenue and above the marketplace freight share; the largest by total freight are {w_txt}."
else:
    w_sentence = "No category is both above-median in revenue and above the marketplace freight share."

In [11]:
print("=== COPY THIS INTO business_questions.md (Q4 answer) ===")
print(f"- Answer: Across delivered orders, freight equals {o.freight_pct_of_price}% of item price overall. "
      f"Among the {len(e)} categories with at least 100 delivered orders, the highest freight shares are {fr_txt}. "
      f"The three largest categories by revenue are {rev_txt}. "
      f"{w_sentence} "
      f"These categories are candidates for shipping-cost, pricing or seller changes.")
print()
print("=== COPY THIS INTO README.md (under ## Method) ===")
print("- Category economics use delivered orders only and categories with at least 100 delivered orders. Freight share of price = total freight / total item price per category. Categories without an English translation use their Portuguese name, and items with no category are labelled 'unknown'. The data has no product costs, so this shows where shipping weighs on customers, not true profit.")

=== COPY THIS INTO business_questions.md (Q4 answer) ===
- Answer: Across delivered orders, freight equals 16.63% of item price overall. Among the 52 categories with at least 100 delivered orders, the highest freight shares are christmas_supplies (36.52% of price, average item price 58.25), signaling_and_security (30.39% of price, average item price 108.2), electronics (29.46% of price, average item price 56.81). The three largest categories by revenue are health_beauty (9.33% of revenue, freight 14.51% of price), watches_gifts (8.82% of revenue, freight 8.42% of price), bed_bath_table (7.74% of revenue, freight 19.72% of price). 12 categories are both above-median in revenue and above the marketplace freight share; the largest by total freight are bed_bath_table (revenue 1,023,435, freight 19.72% of price), furniture_decor (revenue 711,928, freight 23.65% of price), sports_leisure (revenue 954,853, freight 17.11% of price). These categories are candidates for shipping-cost, pricing or